# Robot Referee - Training Notebook

**STATUS:** Active - Colab Training Pipeline  
**PURPOSE:** Train YOLOv8 models with cost optimization (Early Stopping, Mixed Precision)  
**NEXT_STEPS:**
- Implement custom data augmentation strategies
- Add hyperparameter tuning with Optuna
- Set up model versioning and experiment tracking

---

## Cost Optimization Features
- ✅ Early Stopping to prevent overtraining
- ✅ Mixed Precision (AMP) for faster training
- ✅ Efficient batch sizing
- ✅ Model checkpointing to resume training

## 1. Setup and Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python numpy torch torchvision pandas matplotlib seaborn pyyaml

In [ ]:
# Mount Google Drive (optional - for dataset and model storage)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import libraries
import os
import yaml
import torch
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")

## 2. Configuration

In [ ]:
# Training configuration with cost optimization
CONFIG = {
    # Model settings
    "model_type": "yolov8n-pose",  # Options: yolov8n-pose, yolov8s-pose, yolov8m-pose
    
    # Training parameters
    "epochs": 100,
    "batch_size": 16,  # Adjust based on GPU memory
    "img_size": 640,
    
    # Cost optimization
    "patience": 10,  # Early stopping patience (epochs without improvement)
    "amp": True,     # Mixed precision training (Automatic Mixed Precision)
    
    # Data
    "data_yaml": "./dataset/data.yaml",  # Path to dataset config
    "workers": 8,    # Number of dataloader workers
    
    # Output
    "project": "robot_referee",
    "name": "training_run",
    
    # Device
    "device": 0,  # GPU device (0 for cuda:0, 'cpu' for CPU)
}

print("Training Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Dataset Preparation

In [ ]:
# Create data.yaml if it doesn't exist
# This is a template - adjust paths and classes based on your dataset

dataset_yaml_content = """
# Dataset configuration for Robot Referee
# Adjust paths based on your dataset structure

path: ./dataset  # Dataset root directory
train: images/train  # Train images (relative to 'path')
val: images/val      # Val images (relative to 'path')
test: images/test    # Test images (optional)

# Classes
names:
  0: player
  1: ball
  2: referee

# Keypoints for pose detection (if using pose model)
kpt_shape: [17, 3]  # 17 keypoints, 3 values (x, y, visibility)
"""

# Save data.yaml
os.makedirs("./dataset", exist_ok=True)
with open("./dataset/data.yaml", "w") as f:
    f.write(dataset_yaml_content)

print("Dataset configuration created at: ./dataset/data.yaml")
print("\n⚠️ IMPORTANT: Update data.yaml with your actual dataset paths and classes!")

In [ ]:
# Verify dataset structure
dataset_path = Path("./dataset")
if dataset_path.exists():
    print("Dataset directory structure:")
    for item in dataset_path.rglob("*"):
        if item.is_dir():
            print(f"  📁 {item.relative_to(dataset_path)}")
else:
    print("⚠️ Dataset directory not found!")
    print("Please upload your dataset to ./dataset/")

## 4. Model Training with Cost Optimization

In [ ]:
# Load pre-trained model
model = YOLO(f"{CONFIG['model_type']}.pt")

print(f"Loaded model: {CONFIG['model_type']}")
print(f"Model summary:")
model.info()

In [ ]:
# Start training with optimization features
print("\n" + "="*50)
print("Starting Training with Cost Optimizations")
print("="*50)
print(f"✅ Early Stopping: Enabled (patience={CONFIG['patience']})")
print(f"✅ Mixed Precision (AMP): {'Enabled' if CONFIG['amp'] else 'Disabled'}")
print(f"✅ Batch Size: {CONFIG['batch_size']}")
print("="*50 + "\n")

# Train the model
results = model.train(
    data=CONFIG["data_yaml"],
    epochs=CONFIG["epochs"],
    batch=CONFIG["batch_size"],
    imgsz=CONFIG["img_size"],
    patience=CONFIG["patience"],  # Early stopping
    amp=CONFIG["amp"],            # Mixed precision
    workers=CONFIG["workers"],
    project=CONFIG["project"],
    name=CONFIG["name"],
    device=CONFIG["device"],
    save=True,
    save_period=10,  # Save checkpoint every 10 epochs
    plots=True,      # Generate training plots
    verbose=True
)

print("\n✅ Training completed!")

## 5. Training Results Analysis

In [ ]:
# Load and display training results
results_dir = Path(f"{CONFIG['project']}/{CONFIG['name']}")
results_csv = results_dir / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # Remove whitespace from column names
    
    print("Training Results Summary:")
    print(df.tail(10))
    
    # Plot training curves
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss')
    axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss')
    axes[0, 0].set_title('Box Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # mAP
    axes[0, 1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50')
    axes[0, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
    axes[0, 1].set_title('Mean Average Precision')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('mAP')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Precision & Recall
    axes[1, 0].plot(df['epoch'], df['metrics/precision(B)'], label='Precision')
    axes[1, 0].plot(df['epoch'], df['metrics/recall(B)'], label='Recall')
    axes[1, 0].set_title('Precision & Recall')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Value')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Learning rate
    if 'lr/pg0' in df.columns:
        axes[1, 1].plot(df['epoch'], df['lr/pg0'])
        axes[1, 1].set_title('Learning Rate')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('LR')
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(results_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Results file not found.")

## 6. Model Validation

In [ ]:
# Load best model
best_model_path = results_dir / "weights" / "best.pt"

if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    
    # Validate on test set
    val_results = best_model.val(
        data=CONFIG["data_yaml"],
        split="val",
        batch=CONFIG["batch_size"],
        imgsz=CONFIG["img_size"],
        device=CONFIG["device"]
    )
    
    print("\nValidation Results:")
    print(f"  mAP50: {val_results.box.map50:.4f}")
    print(f"  mAP50-95: {val_results.box.map:.4f}")
    print(f"  Precision: {val_results.box.mp:.4f}")
    print(f"  Recall: {val_results.box.mr:.4f}")
else:
    print("Best model not found.")

## 7. Export Model

In [ ]:
# Export model for deployment
if best_model_path.exists():
    # Export to ONNX format (optional - for production deployment)
    onnx_path = best_model.export(format="onnx", imgsz=CONFIG["img_size"])
    print(f"✅ Model exported to ONNX: {onnx_path}")
    
    # Copy best model to models directory
    print(f"\n📦 Best model saved at: {best_model_path}")
    print("\nTo use this model in main.py:")
    print(f"  python main.py --video input.mp4 --pose-model {best_model_path}")

## 8. Cost Analysis

In [ ]:
# Calculate training statistics
if results_csv.exists():
    df = pd.read_csv(results_csv)
    actual_epochs = len(df)
    
    print("\n" + "="*50)
    print("TRAINING COST ANALYSIS")
    print("="*50)
    print(f"Configured epochs: {CONFIG['epochs']}")
    print(f"Actual epochs trained: {actual_epochs}")
    print(f"Early stopping saved: {CONFIG['epochs'] - actual_epochs} epochs")
    
    if actual_epochs < CONFIG['epochs']:
        savings = ((CONFIG['epochs'] - actual_epochs) / CONFIG['epochs']) * 100
        print(f"\n💰 Cost savings: ~{savings:.1f}% of compute time")
    
    print(f"\n✅ Mixed Precision (AMP): {'Enabled' if CONFIG['amp'] else 'Disabled'}")
    if CONFIG['amp']:
        print("   Estimated speedup: 1.5-2x faster training")
    
    print("="*50)

## 9. Download Results (Optional)

In [ ]:
# Compress and download training results
import shutil

if results_dir.exists():
    # Create archive
    archive_name = f"{CONFIG['name']}_results"
    shutil.make_archive(archive_name, 'zip', results_dir)
    
    print(f"✅ Results archived: {archive_name}.zip")
    print("\nDownload the archive to use the trained model locally.")
    
    # Download using Colab files interface
    from google.colab import files
    files.download(f"{archive_name}.zip")